In [2]:
pip install ucimlrepo


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
individual_household_electric_power_consumption = fetch_ucirepo(id=235)

# data (as pandas dataframes)
X = individual_household_electric_power_consumption.data.features
y = individual_household_electric_power_consumption.data.targets

# metadata
print(individual_household_electric_power_consumption.column_names)

# variable information
print(individual_household_electric_power_consumption.variables)



C:\Users\ravis\AppData\Roaming\Python\Python314\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


None
                    name     role         type demographic description units  \
0                   Date  Feature         Date        None        None  None   
1                   Time  Feature  Categorical        None        None  None   
2    Global_active_power  Feature   Continuous        None        None  None   
3  Global_reactive_power  Feature   Continuous        None        None  None   
4                Voltage  Feature   Continuous        None        None  None   
5       Global_intensity  Feature   Continuous        None        None  None   
6         Sub_metering_1  Feature   Continuous        None        None  None   
7         Sub_metering_2  Feature   Continuous        None        None  None   
8         Sub_metering_3  Feature   Continuous        None        None  None   

  missing_values  
0             no  
1             no  
2             no  
3             no  
4             no  
5             no  
6             no  
7             no  
8             no  


In [4]:
X.tail()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
2075254,26/11/2010,20:58:00,0.946,0.0,240.43,4.0,0.0,0.0,0.0
2075255,26/11/2010,20:59:00,0.944,0.0,240.0,4.0,0.0,0.0,0.0
2075256,26/11/2010,21:00:00,0.938,0.0,239.82,3.8,0.0,0.0,0.0
2075257,26/11/2010,21:01:00,0.934,0.0,239.7,3.8,0.0,0.0,0.0
2075258,26/11/2010,21:02:00,0.932,0.0,239.55,3.8,0.0,0.0,0.0


In [5]:
pip install openmeteo_requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
import openmeteo_requests

def fetch_historical_weather(latitude: float, longitude: float,
                             start_date: str, end_date: str,
                             hourly_vars: list = [
                                 "temperature_2m", "relative_humidity_2m",
                                 "wind_speed_10m", "wind_direction_10m"
                             ]):
    """
    Fetch historical hourly weather from the Open-Meteo Archive API.
    """
    client = openmeteo_requests.Client()
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": hourly_vars,
        "timezone": "UTC"
    }
    responses = client.weather_api(
        "https://archive-api.open-meteo.com/v1/archive", params=params
    )
    response = responses[0]
    hourly = response.Hourly()
    hourly_time = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s"),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s"),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    )
    hourly_data = {var: hourly.Variables(i).ValuesAsNumpy() for i, var in enumerate(hourly_vars)}
    return pd.DataFrame(hourly_data, index=hourly_time)

In [7]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

def get_uci_dataset_df(dataset_id):
    """
    Fetch a dataset from the UCI Machine Learning Repository using ucimlrepo
    and return it as a Pandas DataFrame.

    Parameters:
        dataset_id (int): The numeric ID of the dataset in the UCI repository.

    Returns:
        pd.DataFrame: Combined DataFrame with features and targets.
    """
    dataset = fetch_ucirepo(id=dataset_id)

    # Combine features and targets into one DataFrame
    df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

    # Assign column names if available
    if dataset.column_names:
        df.columns = dataset.column_names

    return df


In [8]:
power_df=get_uci_dataset_df(235)


C:\Users\ravis\AppData\Roaming\Python\Python314\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


In [9]:
power_df.tail()


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
2075254,26/11/2010,20:58:00,0.946,0.0,240.43,4.0,0.0,0.0,0.0
2075255,26/11/2010,20:59:00,0.944,0.0,240.0,4.0,0.0,0.0,0.0
2075256,26/11/2010,21:00:00,0.938,0.0,239.82,3.8,0.0,0.0,0.0
2075257,26/11/2010,21:01:00,0.934,0.0,239.7,3.8,0.0,0.0,0.0
2075258,26/11/2010,21:02:00,0.932,0.0,239.55,3.8,0.0,0.0,0.0


In [10]:
weather_df=fetch_historical_weather(latitude=12.9629,longitude=77.5775,start_date="2006-12-16",end_date="2010-11-20")

In [11]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

def normalize_and_join(df_energy, df_weather):
    # --- Step 1: Parse datetime ---
    df_energy['datetime'] = pd.to_datetime(
        df_energy['Date'] + ' ' + df_energy['Time'],
        format='%d/%m/%Y %H:%M:%S',
        errors='coerce'   # invalid formats become NaT
    )
    df_energy = df_energy.drop(columns=['Date', 'Time'])

    df_weather.index = pd.to_datetime(df_weather.index, errors='coerce')
    df_weather = df_weather.reset_index().rename(columns={'index': 'datetime'})

    # --- Step 2: Clean non-numeric values ---
    for col in df_energy.columns:
        if col != 'datetime':
            df_energy[col] = pd.to_numeric(df_energy[col], errors='coerce')
    for col in df_weather.columns:
        if col != 'datetime':
            df_weather[col] = pd.to_numeric(df_weather[col], errors='coerce')

    # Replace missing values with column means (or you can drop them)
    df_energy = df_energy.fillna(df_energy.mean(numeric_only=True))
    df_weather = df_weather.fillna(df_weather.mean(numeric_only=True))

    # --- Step 3: Normalize numeric columns ---
    scaler = MinMaxScaler()

    energy_cols = [c for c in df_energy.columns if c != 'datetime']
    df_energy[energy_cols] = scaler.fit_transform(df_energy[energy_cols])

    weather_cols = [c for c in df_weather.columns if c != 'datetime']
    df_weather[weather_cols] = scaler.fit_transform(df_weather[weather_cols])

    # --- Step 4: Join on nearest timestamp ---
    df_joined = pd.merge_asof(
        df_energy.sort_values('datetime'),
        df_weather.sort_values('datetime'),
        on='datetime',
        direction='nearest'
    )

    return df_joined


In [12]:
merged_df=normalize_and_join(power_df,weather_df)

In [13]:
merged_df['power_lag_1'] = merged_df['Global_active_power'].shift(1)
merged_df['power_lag_24'] = merged_df['Global_active_power'].shift(24)
merged_df['target_power'] = merged_df['Global_active_power']
merged_df.dropna(inplace=True)

In [14]:
merged_df.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,power_lag_1,power_lag_24,target_power
24,0.398153,0.0,0.379968,0.398340,0.0,0.0,0.548387,2006-12-16 17:48:00,0.20229,0.860287,0.314866,0.200262,0.461525,0.374796,0.398153
25,0.287163,0.0,0.434895,0.278008,0.0,0.0,0.548387,2006-12-16 17:49:00,0.20229,0.860287,0.314866,0.200262,0.398153,0.478363,0.287163
26,0.286076,0.0,0.408401,0.278008,0.0,0.0,0.548387,2006-12-16 17:50:00,0.20229,0.860287,0.314866,0.200262,0.287163,0.479631,0.286076
27,0.285352,0.0,0.400646,0.278008,0.0,0.0,0.548387,2006-12-16 17:51:00,0.20229,0.860287,0.314866,0.200262,0.286076,0.480898,0.285352
28,0.288068,0.0,0.397092,0.282158,0.0,0.0,0.548387,2006-12-16 17:52:00,0.20229,0.860287,0.314866,0.200262,0.285352,0.325005,0.288068


In [15]:
# Ensure datetime is the index
merged_df = merged_df.set_index('datetime')

# Add time-based features
merged_df['hour'] = merged_df.index.hour
merged_df['dayofweek'] = merged_df.index.dayofweek   # Monday=0, Sunday=6
merged_df['month'] = merged_df.index.month


In [16]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [17]:
# Train/Test Split (Sequential split for time-series)
train_size = int(len(merged_df) * 0.8)
train, test = merged_df.iloc[:train_size], merged_df.iloc[train_size:]

y_train = train['target_power']
y_test = test['target_power']

# --- BASELINE MODEL (Historical Power + Time Only) ---
features_base = ['power_lag_1', 'power_lag_24', 'hour', 'dayofweek', 'month']
X_train_base = train[features_base]
X_test_base = test[features_base]

model_base = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, objective='reg:squarederror')
model_base.fit(X_train_base, y_train)
preds_base = model_base.predict(X_test_base)

# --- HYBRID MODEL (Baseline + Weather Data) ---
# Notice we are NOT using sub_metering or voltage here, because in a real-world predictive
# scenario, you won't know tomorrow's voltage or sub-metering data in advance.
features_hybrid = ['power_lag_1', 'power_lag_24', 'hour', 'dayofweek', 'month',
                   'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'wind_direction_10m']
X_train_hybrid = train[features_hybrid]
X_test_hybrid = test[features_hybrid]

model_hybrid = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05, objective='reg:squarederror')
model_hybrid.fit(X_train_hybrid, y_train)
preds_hybrid = model_hybrid.predict(X_test_hybrid)

# Evaluation
print(f"Baseline Model RMSE: {np.sqrt(mean_squared_error(y_test, preds_base)):.4f} kW")
print(f"Hybrid Model RMSE:   {np.sqrt(mean_squared_error(y_test, preds_hybrid)):.4f} kW")

Baseline Model RMSE: 0.0194 kW
Hybrid Model RMSE:   0.0194 kW


In [18]:
merged_df

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,power_lag_1,power_lag_24,target_power,hour,dayofweek,month
datetime,,,,,,,,,,,,,,,,,
2006-12-16 17:48:00,0.398153,0.0,0.379968,0.398340,0.0,0.0,0.548387,0.202290,0.860287,0.314866,0.200262,0.461525,0.374796,0.398153,17,5,12
2006-12-16 17:49:00,0.287163,0.0,0.434895,0.278008,0.0,0.0,0.548387,0.202290,0.860287,0.314866,0.200262,0.398153,0.478363,0.287163,17,5,12
2006-12-16 17:50:00,0.286076,0.0,0.408401,0.278008,0.0,0.0,0.548387,0.202290,0.860287,0.314866,0.200262,0.287163,0.479631,0.286076,17,5,12
2006-12-16 17:51:00,0.285352,0.0,0.400646,0.278008,0.0,0.0,0.548387,0.202290,0.860287,0.314866,0.200262,0.286076,0.480898,0.285352,17,5,12
2006-12-16 17:52:00,0.288068,0.0,0.397092,0.282158,0.0,0.0,0.548387,0.202290,0.860287,0.314866,0.200262,0.285352,0.325005,0.288068,17,5,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2010-11-26 20:58:00,0.078762,0.0,0.556704,0.078838,0.0,0.0,0.000000,0.328244,0.966815,0.133536,0.365167,0.078762,0.080753,0.078762,20,4,11
2010-11-26 20:59:00,0.078580,0.0,0.542811,0.078838,0.0,0.0,0.000000,0.328244,0.966815,0.133536,0.365167,0.078762,0.080029,0.078580,20,4,11
2010-11-26 21:00:00,0.078037,0.0,0.536995,0.074689,0.0,0.0,0.000000,0.328244,0.966815,0.133536,0.365167,0.078580,0.080391,0.078037,21,4,11


In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import mean_squared_error

# 1. Scale the Data
# Assuming 'df' is the merged and cleaned dataframe from the previous step
# features_hybrid = ['power_lag_1', 'power_lag_24', 'hour', 'dayofweek', 'month', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'wind_direction_10m']

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(merged_df[features_hybrid])
y_scaled = scaler_y.fit_transform(merged_df[['target_power']])

# 2. Create Sequences for LSTM (e.g., look back 24 hours to predict the next hour)
def create_sequences(X, y, time_steps=24):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

TIME_STEPS = 24
X_seq, y_seq = create_sequences(X_scaled, y_scaled, TIME_STEPS)

# 3. Train/Test Split for LSTM (Sequential)
train_size = int(len(X_seq) * 0.8)
X_train_lstm, X_test_lstm = X_seq[:train_size], X_seq[train_size:]
y_train_lstm, y_test_lstm = y_seq[:train_size], y_seq[train_size:]

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Build the LSTM Model
model_lstm = Sequential([
    LSTM(64, activation='relu', input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2]), return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1) # Output layer for predicting 'target_power'
])

model_lstm.compile(optimizer='adam', loss='mse')

# Train the model
history = model_lstm.fit(
    X_train_lstm, y_train_lstm,
    epochs=2,
    batch_size=32,
    validation_split=0.1,
    verbose=1,
    shuffle=False # Crucial for time-series!
)

# Predict and Inverse Transform to get actual kW values
preds_lstm_scaled = model_lstm.predict(X_test_lstm)
preds_lstm = scaler_y.inverse_transform(preds_lstm_scaled)
y_test_actual = scaler_y.inverse_transform(y_test_lstm)

rmse_lstm = np.sqrt(mean_squared_error(y_test_actual, preds_lstm))
print(f"LSTM Model RMSE: {rmse_lstm:.4f} kW")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/2
10519/46693 ━━━━━━━━━━━━━━━━━━━━ 8:19 14ms/step - loss: 0.0050

KeyboardInterrupt: 